# Testing segmentation models with NN

In [1]:
import laspy
import numpy as np
import open3d as o3d
from pathlib import Path

DATA_PATH = Path("/media/HDD_disk/james/lidar_REALLOCATE")

CITY_FILES = {
    "Riga": DATA_PATH / "riga.laz",
    "Utrecht": DATA_PATH / "utrecht.laz",
    "Vilnius": DATA_PATH / "vilnius.laz",
    "Warsaw": DATA_PATH / "warsaw.laz",
}
city_las = {name: laspy.read(path) for name, path in CITY_FILES.items()}

las = city_las["Riga"]
import numpy as np
import open3d as o3d

xyz = np.c_[las.x, las.y, las.z]

# If your RGB is standard 8-bit: 0..255
rgb = np.c_[las.red, las.green, las.blue].astype(np.float64)

# Normalize to 0..1 for Open3D
if rgb.max() > 1:
    if rgb.max() > 255:
        rgb /= 65535.0   # common for LAS color channels
    else:
        rgb /= 255.0

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
pcd.colors = o3d.utility.Vector3dVector(rgb)



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Make a sample of the point cloud for visualization
pcd = pcd.voxel_down_sample(voxel_size=0.5)

# inspect pointcloud statistics
print(f"Number of points: {len(pcd.points)}")

# Visualise the point cloud using Open3D's built-in visualizer, colour by rgb values
#o3d.visualization.draw_plotly([pcd])

Number of points: 111150


## Simply - let's just try calculating verticality as a threshold

In [5]:
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.3,
        max_nn=30
    )
)
normals = np.asarray(pcd.normals)

verticality = 1 - np.abs(normals[:, 2])

ground_mask = verticality < 0.1
ground_points = points[ground_mask]

NameError: name 'points' is not defined

## Cloth sheet falling

In [3]:
import CSF

xyz = np.asarray(pcd.points)

csf = CSF.CSF()
csf.setPointCloud(xyz)

# parameters tuned for urban TLS
csf.params.cloth_resolution = 0.1
csf.params.rigidness = 3
csf.params.class_threshold = 0.03

ground_idx = CSF.VecInt()
non_ground_idx = CSF.VecInt()

csf.do_filtering(ground_idx, non_ground_idx)

[0] Configuring terrain...
[0]  - bbMin: -57.2436 -39.609 -53.5883
[0]  - bbMax: 38.2385 4.717 59.3706
[0] Configuring cloth...
[0]  - width: 958 height: 1133
[0] Rasterizing...
[0] Simulating...
[0]  - post handle...


In [4]:
ground_points = xyz[list(ground_idx)]

ground_pcd = o3d.geometry.PointCloud()
ground_pcd.points = o3d.utility.Vector3dVector(ground_points)

o3d.visualization.draw_plotly([ground_pcd])